# EDA

`recordlinkage.datasets.load_febrl2(return_links=False)` 

[url](https://recordlinkage.readthedocs.io/en/latest/ref-datasets.html#recordlinkage.datasets.load_febrl2)

The Freely Extensible Biomedical Record Linkage (Febrl) package is distributed with a dataset generator and four datasets generated with the generator. This function returns the second Febrl dataset as a pandas.DataFrame.

> “This data set contains 5000 records (4000 originals and 1000 duplicates), with a maximum of 5 duplicates based on one original record (and a poisson distribution of duplicate records). Distribution of duplicates: 19 originals records have 5 duplicate records 47 originals records have 4 duplicate records 107 originals records have 3 duplicate records 141 originals records have 2 duplicate records 114 originals records have 1 duplicate record 572 originals records have no duplicate record”

**Goal**: Can learned pairwise similarity improve entity resolution under noisy, multi-record entities, and can graph-based refinement resolve inconsistencies in pairwise predictions?

```text
                 FEBRL-2
                    │
          5,000 noisy records
                    │
          ┌─────────┴─────────┐
          │                   │
       entities          corrupted records
          │
          ↓
      Blocking
          ↓
   Candidate pairs
          ↓
     CNN + MLP
          ↓
    pair confidence
          ↓
        Graph
          ↓
 Entity clusters
 ```


In [1]:
import recordlinkage.datasets
import pandas as pd
from funcs import set_env

set_env()

data = recordlinkage.datasets.load_febrl2(return_links=True)
records, links = data[0], data[1]
print(f"DataFrame: records")
print(f"Fields: {records.columns.to_list()}")
print("Description:")
print(records.info())
print("\n")

print(f"Links sie: {links.size}")
print(links[:5])
    

DataFrame: records
Fields: ['given_name', 'surname', 'street_number', 'address_1', 'address_2', 'suburb', 'postcode', 'state', 'date_of_birth', 'soc_sec_id']
Description:
<class 'pandas.DataFrame'>
Index: 5000 entries, rec-2778-org to rec-1143-org
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   given_name     4891 non-null   str   
 1   surname        4936 non-null   str   
 2   street_number  4777 non-null   object
 3   address_1      4891 non-null   str   
 4   address_2      4431 non-null   str   
 5   suburb         4950 non-null   str   
 6   postcode       5000 non-null   object
 7   state          4952 non-null   str   
 8   date_of_birth  4890 non-null   object
 9   soc_sec_id     5000 non-null   object
dtypes: object(4), str(6)
memory usage: 429.7+ KB
None


Links sie: 1934
MultiIndex([(  'rec-63-dup-1',   'rec-63-dup-0'),
            ( 'rec-232-dup-0',    'rec-232-org'),
            ('rec-3123-dup-0',

### First diagnostic

Audit FEBRL-4 specifically for the failure mode I found in previous dataset: whether true duplicates are trivially separable from non-matches by exact field agreement.

In [ ]:
import numpy as np
import pandas as pd

record = records.drop('soc_sec_id')
fields = records.columns.tolist()

# Count exact field agreements for every known positive
positive_exact_matches = []

for link in links:
    org_id, dup_id = link[0], link[1]
    org = records.loc[org_id]
    dup = records.loc[dup_id]

    matches = sum(
        pd.notna(org[field])
        and pd.notna(dup[field])
        and org[field] == dup[field]
        for field in fields
    )

    positive_exact_matches.append(matches)

positive_exact_matches = np.array(positive_exact_matches)

print("Positive exact-field agreement")
print("--------------------------------")
print(f"Mean   : {positive_exact_matches.mean():.2f}")
print(f"Median : {np.median(positive_exact_matches):.0f}")
print(f"Min    : {positive_exact_matches.min()}")
print(f"Max    : {positive_exact_matches.max()}")

print("\nDistribution:")
print(
    pd.Series(positive_exact_matches)
    .value_counts()
    .sort_index()
)

print("\nPercent with >= threshold:")
for threshold in range(1, 9):
    pct = (positive_exact_matches >= threshold).mean()
    print(f">= {threshold}: {pct:.2%}")

Positive exact-field agreement
--------------------------------
Mean   : 6.04
Median : 6
Min    : 1
Max    : 9

Distribution:
1      1
2      7
3     56
4    218
5    371
6    506
7    473
8    292
9     10
Name: count, dtype: int64

Percent with >= threshold:
>= 1: 100.00%
>= 2: 99.95%
>= 3: 99.59%
>= 4: 96.69%
>= 5: 85.42%
>= 6: 66.24%
>= 7: 40.07%
>= 8: 15.62%


In [ ]:
rng = np.random.default_rng(42)
fields = records.columns.tolist()

# Put links into a set so we can quickly identify true matches
true_links = set(links.tolist())
record_ids = records.index.to_list()
negative_exact_matches = []

while len(negative_exact_matches) < len(links):
    id1, id2 = rng.choice(record_ids, size=2, replace=False)
    if (id1, id2) in true_links or (id2, id1) in true_links:
        continue
    r1 = records.loc[id1]
    r2 = records.loc[id2]

    matches = sum(
        pd.notna(r1[field])
        and pd.notna(r2[field])
        and r1[field] == r2[field]
        for field in fields
    )

    negative_exact_matches.append(matches)

negative_exact_matches = np.array(negative_exact_matches)

print("Negative exact-field agreement")
print("--------------------------------")
print(f"Mean   : {negative_exact_matches.mean():.2f}")
print(f"Median : {np.median(negative_exact_matches):.0f}")
print(f"Min    : {negative_exact_matches.min()}")
print(f"Max    : {negative_exact_matches.max()}")

print("\nDistribution:")
print(
    pd.Series(negative_exact_matches)
    .value_counts()
    .sort_index()
)

print("\nPercent with >= threshold:")
for threshold in range(1, 9):
    pct = (negative_exact_matches >= threshold).mean()
    print(f">= {threshold}: {pct:.2%}")

Negative exact-field agreement
--------------------------------
Mean   : 0.24
Median : 0
Min    : 0
Max    : 2

Distribution:
0    1477
1     449
2       8
Name: count, dtype: int64

Percent with >= threshold:
>= 1: 23.63%
>= 2: 0.41%
>= 3: 0.00%
>= 4: 0.00%
>= 5: 0.00%
>= 6: 0.00%
>= 7: 0.00%
>= 8: 0.00%


> Again... random negatives are useless for evaluating the matcher.


### hard negatives

Pairs from different entities that nevertheless have many fields in common.

```python
# Run approx 25min
import networkx as nx
from tqdm import tqdm


fields = records.columns.tolist()

# Entity graph from the FEBRL linkage
G = nx.Graph()
G.add_nodes_from(records.index)
for a, b in links:
    G.add_edge(a, b)
    
entity_id = {}
for entity_idx, component in enumerate(nx.connected_components(G)):
    for record_id in component:
        entity_id[record_id] = entity_idx
entity_id = pd.Series(entity_id)

print(f"Number of entities: {entity_id.nunique():,}")

# Convert fields to strings while preserving missingness
X = records[fields].copy()
X = X.fillna("__MISSING__")

# Find hardest cross-entity pairs
record_ids = records.index.to_numpy()
hard_negatives = []

for i in tqdm(range(len(record_ids))):
    id1 = record_ids[i]
    row1 = X.loc[id1].to_numpy()
    # Compare against records after i
    for j in range(i + 1, len(record_ids)):
        id2 = record_ids[j]
        if entity_id[id1] == entity_id[id2]:
            continue
        row2 = X.loc[id2].to_numpy()
        exact_matches = np.sum(row1 == row2)
        hard_negatives.append((id1, id2, exact_matches))

hard_negatives = pd.DataFrame(hard_negatives, columns=["record_id_1", "record_id_2", "exact_matches"])

print("\nHard negative candidates:")
print(f"Count: {len(hard_negatives):,}")

print("\nDistribution:")
print(
    hard_negatives["exact_matches"]
    .value_counts()
    .sort_index()
)

print("\nTop hardest negatives:")
print(
    hard_negatives
    .sort_values("exact_matches", ascending=False)
    .head(30)
)
```

```console
Number of entities: 4,000
100%|██████████| 5000/5000 [21:10<00:00,  3.94it/s]  

Hard negative candidates:
Count: 12,495,566

Distribution:
exact_matches
0    9351748
1    3033620
2     108576
3       1613
4          9
Name: count, dtype: int64

Top hardest negatives:
             record_id_1     record_id_2  exact_matches
12059380  rec-3622-dup-0    rec-3570-org              4
47400       rec-3808-org    rec-3177-org              4
393765      rec-1874-org    rec-1273-org              4
1428300     rec-2249-org    rec-3445-org              4
1109399     rec-2416-org    rec-3830-org              4
6750168   rec-3447-dup-0    rec-2204-org              4
3680377     rec-3083-org    rec-1650-org              4
3374472   rec-3314-dup-1    rec-2585-org              4
3115986     rec-3830-org    rec-50-dup-1              4
10066401   rec-113-dup-0     rec-647-org              3
10932363     rec-436-org    rec-2051-org              3
783757      rec-88-dup-0    rec-2060-org              3
8841077    rec-599-dup-0    rec-2012-org              3
513976      rec-1166-org    rec-1698-org              3
12172685    rec-1954-org  rec-3677-dup-0              3
772398    rec-3752-dup-0    rec-50-dup-1              3
11132800    rec-2341-org    rec-2079-org              3
1535987      rec-488-org  rec-2916-dup-1              3
9093793     rec-1522-org    rec-3847-org              3
8814677     rec-3666-org    rec-2088-org              3
11097836    rec-3497-org    rec-3557-org              3
10831690     rec-523-org     rec-124-org              3
523672      rec-2131-org    rec-1686-org              3
10429277    rec-2171-org    rec-3126-org              3
8731027        rec-6-org     rec-465-org              3
9434863   rec-1685-dup-1    rec-2581-org              3
12105287   rec-956-dup-2    rec-3846-org              3
11680122  rec-3996-dup-1  rec-2934-dup-0              3
11624074     rec-607-org     rec-572-org              3
12347454    rec-2198-org    rec-3887-org              3
...
```

Same entity:
- 2–9 exact fields
- mean = 6.91

Different entities:
- maximum = 4 exact fields
- only 9 pairs have 4
- 1,613 pairs have 3

The distributions overlap at 3 and 4. That is already substantially better than your original dataset, where the positive and negative distributions had a huge gap. 

Most importantly: **The hardest negative is only 4/10 exact matches.** so the CNN/MLP is a meaningful approach. In comparison with the previous dataset (SPIDER v2), FEBRL-2 gives:
1. real corruption
1. variable corruption severity
1. multiple records per entity
1. duplicate-to-duplicate links
1. 4,000 latent entities
1. meaningful hard negatives
1. a natural graph-clustering problem

### Data Preprocessing

In [12]:
import recordlinkage.datasets
import pandas as pd

data = recordlinkage.datasets.load_febrl2(return_links=True)
records, links = data[0], data[1]

In [13]:
records = records.drop(columns='soc_sec_id') # 'soc_sec_id' causes almost always an exact-match for true pairs (89 variations out of 4000 entities)
num_fields = ['street_number', 'postcode', 'date_of_birth']

# Overall the data looks clean enough
for field in records.columns:
    records[field] = records[field].astype(str) # str casting for blockers
    records[field] = records[field].str.lower().str.strip()
    if field in num_fields:
        records[field] = records[field].str.zfill(int(max(records[field].str.len()))) # Homogenize lengths
records.fillna("_MISSING_", inplace=True) # Several missing values but I don't want to drop them, rather make them explicitly show they are missing

records.sample(3)

,given_name,surname,street_number,address_1,address_2,suburb,postcode,state,date_of_birth
rec_id,,,,,,,,,
rec-523-org,april,nguyen,00002,perry drive,riana,altona meadows,7307,nsw,19720116
rec-1913-org,bianca,laing,00002,fleetwood smith street,entwood,ringwood,2360,nsw,19180107
rec-2321-dup-3,white,jasmine,00006,appel cerscent,rothwell,barcaldine,2160,qld,19210208


### Build an entity map

In [ ]:
data = records.copy()
parent = {rec_id: rec_id for rec_id in data.index}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        parent[root_b] = root_a

for rec_a, rec_b in true_pairs:
    union(rec_a, rec_b)

roots = {rec_id: find(rec_id) for rec_id in data.index} # Compress paths
# Convert arbitrary roots into clean integer entity IDs
root_to_entity = {root: entity_idx for entity_idx, root in enumerate(sorted(set(roots.values())))}
entity_map = pd.Series({rec_id: root_to_entity[root] for rec_id, root in roots.items()}, name="entity_id")

entity_map

rec-2778-org     1977
rec-712-dup-0    3682
rec-1321-org      360
rec-3004-org     2230
rec-1384-org      429
                 ... 
rec-1487-org      543
rec-1856-org      953
rec-3307-org     2566
rec-227-org      1413
rec-1143-org      162
Name: entity_id, Length: 5000, dtype: int64

In [ ]:
entity_map.value_counts()

# Test ETL file

In [1]:
from etl import get_data

records, true_pairs, entity_map = get_data()
records.sample(3)

,given_name,surname,street_number,address_1,address_2,suburb,postcode,state,date_of_birth
rec_id,,,,,,,,,
rec-2135-dup-1,jessica,smallacombe,00066,notting court,moonabba,_MISSING_,2077,sa,19420703
rec-1661-org,liam,fullgrabe,00011,saunders street,immanuel garden,redbank plains,2031,nsw,19350606
rec-183-org,chelsea,cowell,00006,bee place,garanga,albury,6152,vic,19150410


In [2]:
true_pairs[:3]

MultiIndex([(  'rec-63-dup-1', 'rec-63-dup-0'),
            ( 'rec-232-dup-0',  'rec-232-org'),
            ('rec-3123-dup-0', 'rec-3123-org')],
           )

In [3]:
entity_map.sample(3)

rec-1445-org     497
rec-3449-org    2723
rec-3926-org    3253
Name: entity_id, dtype: int64

# Blocking test

In [1]:
from blocking import train_test_sets

data, train_pairs, test_pairs, val_pairs = train_test_sets()
data.sample(3)

Loading train Candidate Pairs from file...
Loading test Candidate Pairs from file...
Loading val Candidate Pairs from file...


,given_name,surname,street_number,address_1,address_2,suburb,postcode,state,date_of_birth
rec_id,,,,,,,,,
rec-2436-dup-0,lucast,fenwick,00030,yanda strceet,st john of god hospital,maclean,4217,nsw,19900518
rec-789-dup-1,pastars,charlotte,_MISSING_,ligertwood street,tarowng,tweed heads south,2760,qld,19560402
rec-3056-org,mitchell,franklin-browne,00011,bunton place,bunkers hill,modbury heights,3942,qld,19810320


In [8]:
names = ["Train", "Test", "Val"]
pairs = [train_pairs, test_pairs, val_pairs]
for name, pair in zip(names, pairs):
    print(f"{name} Sample: \n{pair.sample(3)}\n")

Train Sample: 
             rec_id_1        rec_id_2  label
1405   rec-2496-dup-3  rec-2686-dup-0      0
95       rec-1351-org    rec-2098-org      0
11937     rec-472-org   rec-954-dup-1      0

Test Sample: 
          rec_id_1        rec_id_2  label
80  rec-1252-dup-0  rec-2130-dup-0      0
3     rec-1117-org    rec-50-dup-0      0
31    rec-1930-org     rec-271-org      0

Val Sample: 
           rec_id_1      rec_id_2  label
774   rec-796-dup-0   rec-796-org      1
724  rec-3054-dup-1  rec-3054-org      1
650  rec-3428-dup-0   rec-921-org      0



In [11]:
for name, pair in zip(names, pairs):
    positive_pairs = pair[pair['label']==1]
    pos_ratio = len(positive_pairs)/len(pair)
    print(f"{name} positive ratio: {pos_ratio:.2f}")

Train positive ratio: 0.11
Test positive ratio: 0.56
Val positive ratio: 0.33


In [15]:
train_records = set(train_pairs["rec_id_1"]) | set(train_pairs["rec_id_2"])
test_records  = set(test_pairs["rec_id_1"]) | set(test_pairs["rec_id_2"])
val_records  = set(val_pairs["rec_id_1"]) | set(val_pairs["rec_id_2"])

traintestoverlap = train_records & test_records
testvaloverlap = test_records & val_records
trainvaloverlap = train_records & val_records

print("Train-test Overlap:", len(traintestoverlap))
print("Test-val Overlap:", len(testvaloverlap))
print("Train-val Overlap:", len(trainvaloverlap))

Train-test Overlap: 0
Test-val Overlap: 0
Train-val Overlap: 0


In [19]:
# ============================================================
# 3. FIELD-LEVEL SIMILARITY DIAGNOSTICS
# ============================================================
fields = data.columns.to_list()
data.reset_index(inplace=True)

def pair_diagnostics(data, pairs, fields):
    data_idx = data.set_index("rec_id")

    a = data_idx.loc[pairs["rec_id_1"]]
    b = data_idx.loc[pairs["rec_id_2"]]

    diagnostics = pairs[["rec_id_1","rec_id_2","label"]].copy()

    for field in fields:
        diagnostics[f"{field}_same"] = (a[field].astype(str).values == b[field].astype(str).values)

    same_cols = [f"{field}_same" for field in fields]
    diagnostics["num_exact_matches"] = (diagnostics[same_cols].sum(axis=1))

    return diagnostics

train_dataset = data.join(pd.DataFrame(train_records, columns=['rec_id']).set_index('rec_id'), how='left')
train_diag = pair_diagnostics(train_dataset, train_pairs, fields)

val_dataset = data.join(pd.DataFrame(val_records, columns=['rec_id']).set_index('rec_id'), how='left')
val_diag = pair_diagnostics(val_dataset, val_pairs, fields)

test_dataset = data.join(pd.DataFrame(test_records, columns=['rec_id']).set_index('rec_id'), how='left')
test_diag = pair_diagnostics(test_dataset, test_pairs, fields)

# ---- Summary by label ----

for name, diag in [
    ("TRAIN", train_diag),
    ("VALIDATION", val_diag),
    ("TEST", test_diag)
]:

    print("\n" + "=" * 70)
    print(f"{name} — EXACT FIELD MATCHES")
    print("=" * 70)

    print(
        diag.groupby("label")["num_exact_matches"]
        .describe()
        .round(3)
    )
    


TRAIN — EXACT FIELD MATCHES
         count   mean    std  min  25%  50%  75%  max
label                                                
0      11961.0  0.696  0.685  0.0  0.0  1.0  1.0  4.0
1       1447.0  6.221  1.339  1.0  5.0  6.0  7.0  9.0

VALIDATION — EXACT FIELD MATCHES
       count   mean    std  min  25%  50%  75%  max
label                                              
0      722.0  0.695  0.665  0.0  0.0  1.0  1.0  3.0
1      355.0  6.394  1.274  3.0  6.0  6.0  7.0  9.0

TEST — EXACT FIELD MATCHES
       count   mean    std  min  25%  50%  75%  max
label                                              
0       64.0  1.000  0.756  0.0  1.0  1.0  1.0  4.0
1       82.0  6.561  1.067  4.0  6.0  7.0  7.0  9.0


In [20]:
from sklearn.metrics import classification_report, f1_score


for name, diag in [
    ("TRAIN", train_diag),
    ("VALIDATION", val_diag),
    ("TEST", test_diag)
]:

    print("\n" + "=" * 70)
    print(f"{name} — EXACT FIELD COUNT BASELINE")
    print("=" * 70)

    for threshold in [1, 2, 3, 4, 5, 6, 8, 10]:

        predictions = (
            diag["num_exact_matches"] >= threshold
        ).astype(int)

        labels = diag["label"].values

        f1 = f1_score(
            labels,
            predictions,
            zero_division=0
        )

        print(
            f"Threshold >= {threshold:2d} | "
            f"F1 = {f1:.4f}"
        )


TRAIN — EXACT FIELD COUNT BASELINE
Threshold >=  1 | F1 = 0.2968
Threshold >=  2 | F1 = 0.6735
Threshold >=  3 | F1 = 0.9750
Threshold >=  4 | F1 = 0.9874
Threshold >=  5 | F1 = 0.9415
Threshold >=  6 | F1 = 0.8250
Threshold >=  8 | F1 = 0.3243
Threshold >= 10 | F1 = 0.0000

VALIDATION — EXACT FIELD COUNT BASELINE
Threshold >=  1 | F1 = 0.6283
Threshold >=  2 | F1 = 0.8976
Threshold >=  3 | F1 = 0.9986
Threshold >=  4 | F1 = 0.9929
Threshold >=  5 | F1 = 0.9559
Threshold >=  6 | F1 = 0.8676
Threshold >=  8 | F1 = 0.3565
Threshold >= 10 | F1 = 0.0000

TEST — EXACT FIELD COUNT BASELINE
Threshold >=  1 | F1 = 0.7700
Threshold >=  2 | F1 = 0.9266
Threshold >=  3 | F1 = 0.9939
Threshold >=  4 | F1 = 0.9939
Threshold >=  5 | F1 = 0.9814
Threshold >=  6 | F1 = 0.9139
Threshold >=  8 | F1 = 0.3093
Threshold >= 10 | F1 = 0.0000


In [29]:
data, train_pairs, test_pairs, val_pairs = train_test_sets()

Generating Candidate Pairs using Rule-based and LSH Blocking for train...
Generating Candidate Pairs using Rule-based and LSH Blocking for test...
Generating Candidate Pairs using Rule-based and LSH Blocking for val...


In [30]:
def analyze_candidate_difficulty(pairs, data, name):
    pairs = pairs.copy()

    a = data.loc[pairs["rec_id_1"]].reset_index(drop=True)
    b = data.loc[pairs["rec_id_2"]].reset_index(drop=True)

    fields = [
        "given_name",
        "surname",
        "street_number",
        "address_1",
        "address_2",
        "suburb",
        "postcode",
        "state",
        "date_of_birth",
    ]

    exact_counts = []

    for i in range(len(pairs)):
        count = sum(
            a.loc[i, field] == b.loc[i, field]
            for field in fields
        )
        exact_counts.append(count)

    pairs["exact_matches"] = exact_counts

    print(f"\n{'='*70}")
    print(name)
    print(f"{'='*70}")

    print("\nPositive:")
    print(
        pairs[pairs["label"] == 1]["exact_matches"]
        .value_counts()
        .sort_index()
    )

    print("\nNegative:")
    print(
        pairs[pairs["label"] == 0]["exact_matches"]
        .value_counts()
        .sort_index()
    )
    
analyze_candidate_difficulty(
    train_pairs,
    data,
    "TRAIN"
)

analyze_candidate_difficulty(
    val_pairs,
    data,
    "VALIDATION"
)

analyze_candidate_difficulty(
    test_pairs,
    data,
    "TEST"
)


TRAIN

Positive:
exact_matches
1      1
2      3
3     35
4    131
5    299
6    408
7    446
8    304
9     10
Name: count, dtype: int64

Negative:
exact_matches
0    6580
1    7391
2    1807
3      96
4       1
Name: count, dtype: int64

VALIDATION

Positive:
exact_matches
4    15
5    28
6    52
7    51
8    40
9     3
Name: count, dtype: int64

Negative:
exact_matches
0    293
1    372
2     80
3      1
Name: count, dtype: int64

TEST

Positive:
exact_matches
4     8
5    10
6    13
7    17
8    15
Name: count, dtype: int64

Negative:
exact_matches
0    36
1    71
2    33
Name: count, dtype: int64


Blocker has made downstream probleam very easy

In [ ]:
from itertools import combinations
from collections import Counter
import pandas as pd
from math import comb
from tqdm import tqdm


# ============================================================
# CONFIG
# ============================================================

FIELDS = [
    "given_name",
    "surname",
    "street_number",
    "address_1",
    "address_2",
    "suburb",
    "postcode",
    "state",
    "date_of_birth",
]


# ============================================================
# HELPERS
# ============================================================

def normalize_value(value):
    """
    Normalize values for exact comparison.

    Treat NaN/None as missing. Missing == missing should NOT
    count as an exact field match.
    """
    if pd.isna(value):
        return None

    return str(value).strip().lower()


def build_true_pair_set(true_pairs):
    """
    Convert FEBRL-2 true_pairs MultiIndex into a normalized set
    of unordered record pairs.
    """
    return {
        tuple(sorted((rec_a, rec_b)))
        for rec_a, rec_b in true_pairs
    }


# ============================================================
# MAIN ANALYSIS
# ============================================================

def analyze_full_pair_universe(data, true_pairs, entity_map, fields=FIELDS):
    """
    Calculate exact-field-match distributions across the entire
    FEBRL-2 pair universe.

    Positive:
        pairs contained in true_pairs

    Negative:
        every pair of records belonging to different entities

    Assumes:
        - data.index contains rec_ids
        - true_pairs contains the ground-truth linkage pairs
        - entity membership is derived from true_pairs
    """

    # --------------------------------------------------------
    # Build lookup for normalized field values
    # --------------------------------------------------------

    records = {}

    for rec_id, row in data.iterrows():
        records[rec_id] = {
            field: normalize_value(row[field])
            for field in fields
        }

    # --------------------------------------------------------
    # Ground-truth positive pairs
    # --------------------------------------------------------

    true_pair_set = build_true_pair_set(true_pairs)

    # --------------------------------------------------------
    # Positive distribution
    # --------------------------------------------------------

    positive_counts = Counter()

    for rec_a, rec_b in true_pair_set:

        matches = 0

        for field in fields:
            value_a = records[rec_a][field]
            value_b = records[rec_b][field]

            # Missing values do not count as agreement
            if value_a is not None and value_b is not None:
                if value_a == value_b:
                    matches += 1

        positive_counts[matches] += 1

    # --------------------------------------------------------
    # Negative distribution
    # --------------------------------------------------------
    #
    # We iterate over ALL possible record pairs.
    # A pair is negative if the two records belong to
    # different ground-truth entities.
    #
    # This gives the complete FEBRL-2 pair universe.
    # --------------------------------------------------------

    negative_counts = Counter()

    record_ids = list(data.index)

    total_pairs = len(record_ids) * (len(record_ids) - 1) // 2
    positive_total = len(true_pair_set)

    negative_total = total_pairs - positive_total

    processed = 0

    total_pairs = comb(len(record_ids), 2)

    for rec_a, rec_b in tqdm(combinations(record_ids, 2), total=total_pairs, desc="Processing pairs"):

        # Same entity = positive
        if entity_map[rec_a] == entity_map[rec_b]:
            continue

        matches = 0

        for field in fields:
            value_a = records[rec_a][field]
            value_b = records[rec_b][field]

            if value_a is not None and value_b is not None:
                if value_a == value_b:
                    matches += 1

        negative_counts[matches] += 1

        processed += 1

    # --------------------------------------------------------
    # Create complete distribution
    # --------------------------------------------------------

    max_matches = len(fields)

    distribution = pd.DataFrame({
        "exact_matches": range(max_matches + 1),
        "positive": [
            positive_counts.get(i, 0)
            for i in range(max_matches + 1)
        ],
        "negative": [
            negative_counts.get(i, 0)
            for i in range(max_matches + 1)
        ],
    })

    distribution["total"] = (
        distribution["positive"] +
        distribution["negative"]
    )

    distribution["positive_pct"] = (
        distribution["positive"] /
        positive_total * 100
    )

    distribution["negative_pct"] = (
        distribution["negative"] /
        negative_total * 100
    )

    # --------------------------------------------------------
    # Print results
    # --------------------------------------------------------

    print("=" * 70)
    print("FULL FEBRL-2 PAIR UNIVERSE")
    print("=" * 70)

    print(f"Records              : {len(record_ids):,}")
    print(f"Possible pairs       : {total_pairs:,}")
    print(f"Positive pairs       : {positive_total:,}")
    print(f"Negative pairs       : {negative_total:,}")
    print(f"Processed negatives  : {processed:,}")

    print("\n" + "=" * 70)
    print("EXACT FIELD-MATCH DISTRIBUTION")
    print("=" * 70)

    print(distribution.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    # --------------------------------------------------------
    # Sanity checks
    # --------------------------------------------------------

    assert positive_counts.total() == positive_total
    assert negative_counts.total() == negative_total

    print("\n" + "=" * 70)
    print("SANITY CHECKS")
    print("=" * 70)

    print(
        f"Positive distribution total: "
        f"{positive_counts.total():,}"
    )

    print(
        f"Negative distribution total: "
        f"{negative_counts.total():,}"
    )

    print(
        f"All pairs accounted for: "
        f"{positive_counts.total() + negative_counts.total():,} "
        f"/ {total_pairs:,}"
    )

    # --------------------------------------------------------
    # Hard-negative summary
    # --------------------------------------------------------

    hard_negatives_3 = sum(
        count
        for matches, count in negative_counts.items()
        if matches >= 3
    )

    hard_negatives_4 = sum(
        count
        for matches, count in negative_counts.items()
        if matches >= 4
    )

    print("\n" + "=" * 70)
    print("HARD NEGATIVES")
    print("=" * 70)

    print(
        f"Negative pairs with >= 3 exact matches: "
        f"{hard_negatives_3:,}"
    )

    print(
        f"Negative pairs with >= 4 exact matches: "
        f"{hard_negatives_4:,}"
    )

    return distribution, entity_map

from etl import get_data

records, true_pairs, entity_map = get_data()
distribution, entity_map = analyze_full_pair_universe(data,true_pairs, entity_map)

In [36]:
from etl import get_data

records, true_pairs, entity_map = get_data()
distribution, entity_map = analyze_full_pair_universe(data,true_pairs, entity_map)

Processing pairs: 100%|██████████| 12497500/12497500 [02:04<00:00, 100196.31it/s]

FULL FEBRL-2 PAIR UNIVERSE
Records              : 5,000
Possible pairs       : 12,497,500
Positive pairs       : 1,934
Negative pairs       : 12,495,566
Processed negatives  : 12,495,566

EXACT FIELD-MATCH DISTRIBUTION
 exact_matches  positive  negative   total  positive_pct  negative_pct
             0         0   9351748 9351748        0.0000       74.8405
             1         1   3033618 3033619        0.0517       24.2776
             2         7    108578  108585        0.3619        0.8689
             3        46      1613    1659        2.3785        0.0129
             4       175         9     184        9.0486        0.0001
             5       344         0     344       17.7870        0.0000
             6       475         0     475       24.5605        0.0000
             7       514         0     514       26.5770        0.0000
             8       359         0     359       18.5626        0.0000
             9        13         0      13        0.6722        0.0000
